In [ ]:
import time
notebook_start = time.perf_counter()
# %pip install -e /home/darshan/A6/PCSAFT_cDFT/thermoift

import os
import numpy  as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import thermoift.PLOT_SETTINGS as ps
import seaborn as sns
import feos

from thermoift.FeosPlugin               import (RegistryManager, ParameterBuilder, VLECalculator, CompositionHandler)
from sklearn.preprocessing              import StandardScaler
from sklearn.pipeline                   import Pipeline
from sklearn.gaussian_process           import GaussianProcessRegressor
from sklearn.gaussian_process.kernels   import Matern, ConstantKernel, WhiteKernel

RegistryManager.load_registry()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG  —  GPR_AL_V2 (multi-temperature, A4 dataset, variable u_t, 20 trials)
# ═══════════════════════════════════════════════════════════════════════════════

POOL_PATH = "../POOL/composition_pool.csv"
A4_PATH   = "../../DATASET_A4/CombinedDataset_A4.csv"

# Six reference temperatures — pool is expanded to (composition × T) space
T_GRID    = [200.0, 220.0, 240.0, 260.0, 280.0, 300.0]   # K
T_WINDOW  = 1.0                                          # K  — A4 filter window per T

# Calibrated u_t per N (originally from the single-T A2 calibration).
# In the joint (composition, T) pool, the U-trajectory may differ;
# Cell 7 auto-calibrates and overrides this dict.
U_T_BY_N = {
     10: 2.60e-3,
     20: 6.96e-5,
     30: 1.01e-5,
     40: 3.44e-6,
     50: 1.46e-6,
     60: 6.88e-7,
     70: 4.56e-7,
     80: 3.19e-7,
     90: 1.96e-7,
    100: 1.56e-7,
}
N_SIZES = [25, 50, 75, 100]                # u_t(N) auto-calibrated in Cell 7
N_MAX   = max(N_SIZES)

# 20 independent trial seeds — shared across AL / Random / Stratified
N_TRIALS    = 20
BASE_SEED   = 770077
TRIAL_SEEDS = [BASE_SEED + 1000 * i for i in range(N_TRIALS)]

# GPR fitting hyper-params
GPR_MAX_SAMPLES   = 5000
RESTART_OPTIMIZER = 3
GPR_SEED          = BASE_SEED              # one-time fit; reused across all trials

OUTPUT_DIR = "AL_V2_MT"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"T_GRID    : {T_GRID} K  ({len(T_GRID)} temperatures)")
print(f"T_WINDOW  : ±{T_WINDOW} K")
print(f"Joint pool: 1000 compositions × {len(T_GRID)} T = {1000*len(T_GRID):,} rows")
print(f"N_SIZES   : {N_SIZES}")
print(f"N_TRIALS  : {N_TRIALS}")
print(f"u_t map   : {U_T_BY_N}")


In [ ]:
# ── Column-name bridge — feature vector is (z, T) ∈ R^9 ────────────────────────
CSV_TO_FULL = {
    "CO2" : "carbon dioxide",
    "H2"  : "hydrogen",
    "Ar"  : "argon",
    "N2"  : "nitrogen",
    "CH4" : "methane",
    "O2"  : "oxygen",
    "CO"  : "carbon monoxide",
    "H2S" : "hydrogen sulfide",
}
POOL_Z_COLS = list(CSV_TO_FULL.keys())
Z_FEATURES  = [f"z_{v}" for v in CSV_TO_FULL.values()]
COMP_MAP    = {k: f"z_{v}" for k, v in CSV_TO_FULL.items()}

FEATURES    = Z_FEATURES + ["temperature"]
TARGET      = "P_bubble"
TARGET_UNIT = "bar"

print(f"Pool columns : {POOL_Z_COLS}")
print(f"GPR features : {FEATURES}  ({len(FEATURES)} total)")


In [ ]:
# ── Load candidate pool (compositions only, 1000 rows) ─────────────────────────
pool = pd.read_csv(POOL_PATH)
print(f"Pool loaded: {len(pool):,} compositions")
print(pool.head(3))


In [ ]:
# ── Load A4, filter to 8-component rows near T_GRID, fit joint (z,T) GPR ──────
a4 = pd.read_csv(A4_PATH); a4.columns = [c.strip() for c in a4.columns]

extra_z = [c for c in a4.columns if c.startswith("z_") and c not in Z_FEATURES]
if extra_z:
    a4 = a4[a4[extra_z].sum(axis=1) < 1e-6].copy()
    print(f"8-component rows: {len(a4):,}")

T_mask = np.zeros(len(a4), dtype=bool)
for T in T_GRID:
    T_mask |= np.abs(a4["temperature"] - T) <= T_WINDOW
a4_ref = a4[T_mask].copy()
if len(a4_ref) < 100:
    print(f"Only {len(a4_ref)} rows near T_GRID — using all temperatures")
    a4_ref = a4.copy()

print(f"A4 rows near T_GRID: {len(a4_ref):,}")
for T in T_GRID:
    n = (np.abs(a4_ref["temperature"] - T) <= T_WINDOW).sum()
    print(f"  T={int(T)} K : {n:,} rows")

df_train = (a4_ref[Z_FEATURES + ["temperature", TARGET]]
              .drop_duplicates(subset=Z_FEATURES + ["temperature"])
              .dropna(subset=[TARGET])
              .reset_index(drop=True))
if len(df_train) > GPR_MAX_SAMPLES:
    df_train = df_train.sample(n=GPR_MAX_SAMPLES, random_state=GPR_SEED).reset_index(drop=True)

print(f"\nGPR training set : {len(df_train):,} labelled (composition, T) rows")
print(f"P_bubble range   : {df_train[TARGET].min():.2f} – {df_train[TARGET].max():.2f} bar")

n_feat = len(FEATURES)
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(nu=2.5, length_scale=np.ones(n_feat), length_scale_bounds=(1e-3, 1e3))
    + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-4, 100.0))
)
gpr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("gpr",    GaussianProcessRegressor(
        kernel=kernel, alpha=0.0, normalize_y=True,
        n_restarts_optimizer=RESTART_OPTIMIZER, random_state=GPR_SEED)),
])

t0 = time.perf_counter()
gpr_model.fit(df_train[FEATURES], df_train[TARGET])
print(f"\nGPR fitting time : {time.perf_counter()-t0:.1f} s")


In [ ]:
# ── GPR diagnostics + ARD feature importances ─────────────────────────────────
signal_kernel = gpr_model.named_steps["gpr"].kernel_.k1
sigma_f_sq    = float(signal_kernel.k1.constant_value)
scaler        = gpr_model.named_steps["scaler"]

print("Fitted kernel:"); print(f"  {gpr_model.named_steps['gpr'].kernel_}")
print(f"σ_f²* = {sigma_f_sq:.4f}")

ls  = signal_kernel.k2.length_scale
imp = 1.0 / ls;  imp /= imp.sum()
print("\nARD feature importances (1 / length-scale, normalised):")
for name, val in sorted(zip(FEATURES, imp), key=lambda x: -x[1]):
    bar = "█" * max(1, int(val * 40))
    print(f"  {name:<25s}  {val*100:5.1f}%  {bar}")

y_pred, _ = gpr_model.predict(df_train[FEATURES], return_std=True)
res = df_train[TARGET].values - y_pred
print(f"\nTrain RMSE : {np.sqrt(np.mean(res**2)):.3f} bar")
print(f"Train R²   : {1 - np.var(res)/np.var(df_train[TARGET].values):.4f}")


In [ ]:
# ── Expand pool × T_GRID → joint (composition, temperature) pool ──────────────
pool_renamed = pool.rename(columns=COMP_MAP)
blocks = []
for T in T_GRID:
    blk = pool_renamed[Z_FEATURES].copy()
    blk["temperature"] = T
    blocks.append(blk)
pool_joint = pd.concat(blocks, ignore_index=True)

pool_size         = len(pool)
joint_to_pool_idx = np.tile(np.arange(pool_size), len(T_GRID))
joint_to_T        = np.repeat(T_GRID, pool_size)

X_pool = scaler.transform(pool_joint[FEATURES].values)

print(f"Joint pool : {X_pool.shape}   (1000 × {len(T_GRID)} = {len(X_pool):,} rows)")
print(f"k(x,x) mean (first 5): {signal_kernel.diag(X_pool[:5]).mean():.4f}  (≈ σ_f²*)")


In [ ]:
# ── Auto-calibrate u_t(N) on the joint multi-T pool ─────────────────────────
# The U-trajectory differs from the single-T A2 calibration: σ_f²*, the pool
# size and the (z, T) feature space all shift U. We run greedy AL once with
# no early-stop up to N_MAX + 5, record U at every pick, and derive
# u_t(N) = sqrt(U[N] * U[N+1]) so the loop terminates cleanly at |S| = N.

def greedy_trace(X_all, kernel, n_max, seed):
    rng    = np.random.default_rng(seed)
    n_pool = len(X_all)
    s_list = list(rng.choice(n_pool, size=2, replace=False))
    p_set  = set(range(n_pool)) - set(s_list)
    u_pick = [np.nan, np.nan]
    while p_set and len(s_list) < n_max:
        p_list = list(p_set)
        K_TT_d = kernel.diag(X_all[p_list])
        K_TS   = kernel(X_all[p_list], X_all[s_list])
        K_SS   = kernel(X_all[s_list], X_all[s_list]) + 1e-8 * np.eye(len(s_list))
        L      = np.linalg.cholesky(K_SS)
        V      = np.linalg.solve(L, K_TS.T)
        U      = np.maximum(K_TT_d - np.einsum("ij,ij->j", V, V), 0.0)
        best   = int(np.argmax(U))
        s_list.append(p_list[best])
        u_pick.append(float(U[best]))
        p_set.discard(p_list[best])
    return s_list, np.asarray(u_pick)

print("Running one-shot greedy trace on the joint pool for u_t calibration …")
t0 = time.perf_counter()
_, u_trace = greedy_trace(X_pool, signal_kernel, N_MAX + 5, BASE_SEED)
print(f"  trace done in {time.perf_counter()-t0:.1f} s   "
      f"(U at step 2 = {u_trace[2]:.3e},  U at step {N_MAX} = {u_trace[N_MAX]:.3e})")

U_T_BY_N_AUTO = {}
for N in N_SIZES:
    u_lo = u_trace[N]
    u_hi = u_trace[N - 1]
    U_T_BY_N_AUTO[N] = float(np.sqrt(u_lo * u_hi)) if (u_lo > 0 and u_hi > 0) else float(u_lo)

print("\nAuto-calibrated u_t per N:")
print(f"  {"N":>4}  {"u_t (auto)":>14}")
for N in N_SIZES:
    print(f"  {N:>4}  {U_T_BY_N_AUTO[N]:>14.3e}")

U_T_BY_N = U_T_BY_N_AUTO


In [ ]:
# ── Algorithm 1 — kernel-based active learning ────────────────────────────────
def compute_U_T(X_T, X_S, kernel):
    K_TT_diag = kernel.diag(X_T)
    K_TS      = kernel(X_T, X_S)
    K_SS      = kernel(X_S, X_S) + 1e-8 * np.eye(len(X_S))
    L = np.linalg.cholesky(K_SS)
    V = np.linalg.solve(L, K_TS.T)
    return np.maximum(K_TT_diag - np.einsum("ij,ij->j", V, V), 0.0)


def run_algorithm1(X_all, kernel, n_max, u_t, seed, n_t=None):
    rng    = np.random.default_rng(seed)
    n_pool = len(X_all)
    s_list = list(rng.choice(n_pool, size=2, replace=False))
    p_set  = set(range(n_pool)) - set(s_list)

    while p_set and len(s_list) < n_max:
        p_list = list(p_set)
        if n_t is not None and len(p_list) > n_t:
            t_idx  = rng.choice(len(p_list), size=n_t, replace=False)
            t_list = [p_list[i] for i in t_idx]
        else:
            t_list = p_list

        U = compute_U_T(X_all[t_list], X_all[s_list], kernel)
        best = int(np.argmax(U))
        if U[best] > u_t:
            s_list.append(t_list[best])
            p_set.discard(t_list[best])
        for i, idx in enumerate(t_list):
            if U[i] < u_t:
                p_set.discard(idx)
    return s_list


print("Running AL on joint pool: 20 trials × |N_SIZES| runs …")
t0 = time.perf_counter()
al_selections = {}
al_warnings   = []

for tr, seed in enumerate(TRIAL_SEEDS):
    for N in N_SIZES:
        sel = run_algorithm1(
            X_all  = X_pool,
            kernel = signal_kernel,
            n_max  = N,
            u_t    = U_T_BY_N[N],
            seed   = seed,
        )
        if len(sel) != N:
            al_warnings.append((tr, seed, N, len(sel), U_T_BY_N[N]))
        al_selections[(tr, N)] = sel
    if (tr + 1) % 5 == 0:
        print(f"  trial {tr+1:2d}/{N_TRIALS} done")

elapsed_al = time.perf_counter() - t0
print(f"\nAL trials done in {elapsed_al:.1f} s")
if al_warnings:
    print(f"⚠  {len(al_warnings)} (trial, N) pairs missed the target size.")
    for tr, seed, N, got, u in al_warnings[:10]:
        print(f"  trial {tr:2d} seed={seed}  N={N:3d}  got={got:3d}  u_t={u:.2e}")
else:
    print("All AL trials reached the target N exactly. ✓")


In [ ]:
# ── Random baseline: 20 trials on the joint pool ──────────────────────────────
print("Generating random selections from joint pool …")
random_selections = {}
n_joint = len(X_pool)
for tr, seed in enumerate(TRIAL_SEEDS):
    rng  = np.random.default_rng(seed)
    perm = rng.permutation(n_joint)
    for N in N_SIZES:
        random_selections[(tr, N)] = list(perm[:N])
print(f"Random: {N_TRIALS} trials × {len(N_SIZES)} sizes built")


In [ ]:
# ── Stratified baseline: 20 trials, strata = (CO2_bin × mixture_size × T) ────
def stratified_select_joint(n, pool_df, T_grid, j2pidx, j2T, seed):
    rows = []
    for jidx in range(len(j2pidx)):
        r = pool_df.iloc[int(j2pidx[jidx])].to_dict()
        r["temperature"] = float(j2T[jidx])
        r["_joint_idx"]  = jidx
        rows.append(r)
    sp = pd.DataFrame(rows)
    sp["_co2_bin"] = pd.cut(
        sp["CO2"],
        bins=[0.0, 0.94, 0.96, 0.97, 0.98, 0.99, 1.01],
        labels=["<0.94", "0.94-0.96", "0.96-0.97", "0.97-0.98", "0.98-0.99", "≥0.99"],
    )
    counts = sp.groupby(["_co2_bin", "mixture_size", "temperature"], observed=True).size()
    alloc  = (counts / counts.sum() * n).round().clip(lower=0).astype(int)
    while alloc.sum() > n:  alloc[alloc.idxmax()] -= 1
    while alloc.sum() < n:  alloc[alloc.idxmin()] += 1
    frames = []
    for (cb, ms, T), k in alloc.items():
        if k <= 0: continue
        sub = sp[(sp["_co2_bin"] == cb) & (sp["mixture_size"] == ms) & (sp["temperature"] == T)]
        if len(sub):
            frames.append(sub.sample(n=min(k, len(sub)), random_state=seed))
    result = pd.concat(frames).drop(columns=["_co2_bin"]).drop_duplicates("_joint_idx").head(n)
    return list(result["_joint_idx"].astype(int).values)


print("Generating stratified selections from joint pool …")
stratified_selections = {}
for tr, seed in enumerate(TRIAL_SEEDS):
    for N in N_SIZES:
        stratified_selections[(tr, N)] = stratified_select_joint(
            N, pool, T_GRID, joint_to_pool_idx, joint_to_T, seed)
    if (tr + 1) % 5 == 0:
        print(f"  trial {tr+1:2d}/{N_TRIALS} done")
print(f"Stratified: {N_TRIALS} trials × {len(N_SIZES)} sizes built")


In [ ]:
# ── PC-SAFT P_bubble for every unique (pool_idx, T) pair across all trials ───
ALL_COMPS_FULL = [CSV_TO_FULL[k] for k in POOL_Z_COLS]

def compute_pbubble_single(z_arr, T_K):
    active_z, active_comps, _ = CompositionHandler.reduce_components(
        z_arr, ALL_COMPS_FULL, verbose=False)
    params = ParameterBuilder.build_parameters(active_comps)
    func   = feos.HelmholtzEnergyFunctional.pcsaft(params)
    feed   = CompositionHandler.compute_feed_moles(active_z)
    T_bub, P_bub = VLECalculator.compute_bubble_curve(
        func, [T_K], feed, verbose=False)
    return float(P_bub[0]) if len(P_bub) else float("nan")


needed_pairs = set()
for sel_dict in (al_selections, random_selections, stratified_selections):
    for sel in sel_dict.values():
        for ji in sel:
            needed_pairs.add((int(joint_to_pool_idx[ji]), float(joint_to_T[ji])))
needed_pairs = sorted(needed_pairs)
print(f"Unique (composition, T) pairs to evaluate: {len(needed_pairs)}")

pbubble_cache = {}
n_failed = 0
t0 = time.perf_counter()
for i, (pidx, T) in enumerate(needed_pairs):
    row   = pool.iloc[pidx]
    z_arr = np.array([row[k] for k in POOL_Z_COLS])
    try:
        p = compute_pbubble_single(z_arr, T)
    except Exception as e:
        if n_failed < 3:
            print(f"  [warn] pool[{pidx}] T={int(T)} K: {e}")
        p = float("nan"); n_failed += 1
    pbubble_cache[(pidx, T)] = p
    if (i + 1) % 50 == 0:
        print(f"  {i+1:5d}/{len(needed_pairs)}  ({n_failed} failed)")
elapsed_pb = time.perf_counter() - t0
n_ok = sum(1 for v in pbubble_cache.values() if not np.isnan(v))
print(f"\nP_bubble computed: {n_ok}/{len(needed_pairs)} OK, {n_failed} failed")
print(f"Time: {elapsed_pb:.1f} s  ({elapsed_pb/max(len(needed_pairs),1):.2f} s/pair)")


In [ ]:
# ── Build per-(strategy, trial, N) DataFrames ──────────────────────────────────
def make_selection_df(joint_idx_list, pool_df, pbubble_cache, j2pidx, j2T):
    rows = []
    for ji in joint_idx_list:
        pidx = int(j2pidx[ji])
        T    = float(j2T[ji])
        r    = pool_df.iloc[pidx].to_dict()
        r["pool_idx"]    = pidx
        r["temperature"] = T
        r["P_bubble"]    = pbubble_cache.get((pidx, T), float("nan"))
        rows.append(r)
    return pd.DataFrame(rows).reset_index(drop=True)


all_selections = {
    "AL":         al_selections,
    "Random":     random_selections,
    "Stratified": stratified_selections,
}
selection_df = {}
for strat, sel_dict in all_selections.items():
    for (tr, N), idx_list in sel_dict.items():
        selection_df[(strat, tr, N)] = make_selection_df(
            idx_list, pool, pbubble_cache, joint_to_pool_idx, joint_to_T)
print(f"DataFrames built: {len(selection_df)} entries")


In [ ]:
# ── Trial-aggregated metrics (mean ± std over the 20 trials) ─────────────────
def metric(df, name):
    if name == "mean_Pbubble":   return df["P_bubble"].mean()
    if name == "std_CO2":        return df["CO2"].std()
    if name == "median_Pbubble": return df["P_bubble"].median()
    if name == "T_unique":       return df["temperature"].nunique()
    raise ValueError(name)

metric_names = ["mean_Pbubble", "std_CO2", "median_Pbubble", "T_unique"]
strategies   = list(all_selections.keys())

records = []
for strat in strategies:
    for N in N_SIZES:
        for m in metric_names:
            vals = [metric(selection_df[(strat, tr, N)], m) for tr in range(N_TRIALS)]
            records.append({
                "strategy": strat, "N": N, "metric": m,
                "mean":  np.nanmean(vals),
                "std":   np.nanstd(vals, ddof=1) if len(vals) > 1 else 0.0,
                "min":   np.nanmin(vals),
                "max":   np.nanmax(vals),
            })
metrics_df = pd.DataFrame(records)
print(metrics_df.head(15).to_string(index=False))


In [ ]:
# ── Plot: learning curves (mean ± seed-to-seed std) + T allocation ───────────
style = {
    "AL":         ("crimson",    "-",  "o"),
    "Stratified": ("darkorange", "--", "s"),
    "Random":     ("royalblue",  ":",  "^"),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, mname, ylab in zip(
        axes,
        ["mean_Pbubble",        "std_CO2",            "T_unique"],
        ["Mean P_bubble (bar)", "Std(CO2 fraction)",  f"Unique T (out of {len(T_GRID)})"]):
    for strat in strategies:
        sub = (metrics_df[(metrics_df["strategy"] == strat) &
                          (metrics_df["metric"] == mname)]
               .sort_values("N"))
        col, ls, mk = style[strat]
        ax.plot(sub["N"], sub["mean"], color=col, linestyle=ls, lw=2,
                marker=mk, ms=5, label=strat)
        ax.fill_between(sub["N"], sub["mean"] - sub["std"], sub["mean"] + sub["std"],
                        color=col, alpha=0.18, lw=0)
    ax.set_xlabel("Sample budget N")
    ax.set_ylabel(ylab)
    ax.set_title(ylab + "  (20-trial mean ± std)")
    ax.legend(fontsize=9)
    ps.apply_axis_style(ax)
plt.tight_layout()
ps.save_plot(fig, "V2_MT_learning_curves", folder=OUTPUT_DIR)
plt.show()


fig2, ax = plt.subplots(figsize=(9, 4))
T_colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(T_GRID)))
xs = np.arange(len(strategies))
width = 0.8 / len(T_GRID)
for j, (T, c) in enumerate(zip(T_GRID, T_colors)):
    counts = []
    stds   = []
    for strat in strategies:
        per_trial = [(selection_df[(strat, tr, N_MAX)]["temperature"] == T).sum()
                     for tr in range(N_TRIALS)]
        counts.append(np.mean(per_trial))
        stds.append(np.std(per_trial, ddof=1))
    ax.bar(xs + j * width, counts, width=width, yerr=stds, capsize=2,
           color=c, edgecolor="white", linewidth=0.4, label=f"{int(T)} K")
ax.axhline(N_MAX / len(T_GRID), ls="--", color="0.4", lw=1, label="uniform")
ax.set_xticks(xs + width * (len(T_GRID) - 1) / 2)
ax.set_xticklabels(strategies)
ax.set_ylabel(f"Mean count at temperature  (N={N_MAX}, over {N_TRIALS} trials)")
ax.set_title("Temperature allocation in the selected set, by strategy")
ax.legend(fontsize=8, ncol=3)
ps.apply_axis_style(ax)
plt.tight_layout()
ps.save_plot(fig2, "V2_MT_T_allocation", folder=OUTPUT_DIR)
plt.show()


In [ ]:
# ── Save outputs: per-(strategy, trial, N) CSVs + aggregated metrics ─────────
for strat in strategies:
    d = os.path.join(OUTPUT_DIR, strat); os.makedirs(d, exist_ok=True)
    for (s, tr, N), df in selection_df.items():
        if s != strat: continue
        df.to_csv(os.path.join(d, f"{strat}_trial{tr:02d}_N{N:03d}.csv"), index=False)

metrics_df.to_csv(os.path.join(OUTPUT_DIR, "metrics_trial_aggregated.csv"), index=False)
joblib.dump(gpr_model, os.path.join(OUTPUT_DIR, "GPR_AL_V2_MT_model.joblib"))
print(f"Saved {len(selection_df)} per-trial CSVs + metrics + GPR model → {OUTPUT_DIR}")


In [ ]:
elapsed_total = (time.perf_counter() - notebook_start) / 60
print(f"Total notebook runtime : {elapsed_total:.1f} min")
print(f"  AL trials   : {elapsed_al:.1f} s")
print(f"  PC-SAFT     : {elapsed_pb:.1f} s")
